# Zadanie: eksploracyjna analiza danych — `viral_songs.csv`

**Jak pracować z notebookiem**

- Uruchamiaj komórki po kolei, od góry.
- W komórkach oznaczonych **✏️ Zadanie** uzupełnij miejsca `___`. Pozostałe komórki uruchom bez zmian.
- Pod każdym **❓ Pytaniem** dopisz krótką odpowiedź w komórce *Odpowiedź*.
- Dane wczytują się z folderu `dane/` repozytorium, a jeśli go nie ma (np. w Colabie), bezpośrednio z GitHuba.
- Rozwiązanie: `gotowe-notebooki/00_eda.ipynb`.

## 1. Przygotowanie

### Importy

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

### Wczytanie danych

In [4]:
from pathlib import Path

URL = "https://raw.githubusercontent.com/mzntk/ml-seminarium-projekt/main/dane/viral_songs.csv"
LOCAL = Path("../dane/viral_songs.csv")   # ścieżka w sklonowanym repozytorium

df = pd.read_csv(LOCAL if LOCAL.exists() else URL)

FEATURES = [
    "duration_sec", "danceability", "energy", "valence", "tempo_bpm",
    "acousticness", "instrumentalness", "speechiness", "liveness", "loudness_db",
]

df["viral"] = df["viral"].astype(int)

### ✏️ Zadanie 1: przekształcenie logarytmiczne

Utwórz kolumnę `log_streams` = log₁₀(`streams` + 1). Przyda się do wykresów i korelacji.

In [ ]:
df["log_streams"] = ___

### ✏️ Zadanie 2: przegląd danych

Wyświetl wymiary zbioru, typy kolumn, pierwsze wiersze, statystyki opisowe oraz liczbę braków i duplikatów.

In [ ]:
print(df.shape)
df.info()
display(df.head())
display(df[FEATURES + ["streams"]].describe().T)
print("Braki danych:\n", df.isna().sum())
print("Duplikaty:", df.duplicated().sum())

**❓ Pytanie:** Ile utworów i kolumn ma zbiór? Czy są braki lub duplikaty? Które cechy przyjmują wartości z przedziału [0, 1]?

**Odpowiedź:** …

### ✏️ Zadanie 3: funkcja pomocnicza

Funkcja rysuje histogram i nakłada gęstość rozkładu normalnego o tej samej średniej i odchyleniu standardowym co dane. Uzupełnij statystyki i parametry rozkładu.

In [ ]:
def hist_with_normal(ax, x, name, bins=30):
    x = pd.Series(x).dropna()
    mean, var, std = x.___, x.___, x.___

    ax.hist(x, bins=bins, density=True, edgecolor="black")
    grid = np.linspace(x.min(), x.max(), 300)
    ax.plot(grid, stats.norm.pdf(grid, ___), color="red", label="N(μ, σ²)")
    ax.axvline(mean, color="black", ls="--", label="średnia")

    ax.set_title(f"{name}\nśrednia = {mean:.3g}   wariancja = {var:.3g}")
    ax.set_ylabel("Gęstość")
    ax.legend()

## 2. Zmienne objaśniające

### ✏️ Zadanie 4: histogramy cech

Narysuj histogram każdej cechy z listy `FEATURES`.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 22))
for ax, col in zip(axes.flat, ___):
    hist_with_normal(ax, ___, col)
plt.tight_layout()
plt.show()

**❓ Pytanie:** Które cechy mają rozkład zbliżony do normalnego? Które mają wiele wartości dokładnie równych 0 lub 1 i co to może oznaczać?

**Odpowiedź:** …

## 3. Zmienne objaśniane

### ✏️ Zadanie 5: `streams` (model 1)

Porównaj histogram `streams` w oryginalnej skali i po przekształceniu `log10`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hist_with_normal(axes[0], ___, "streams")
hist_with_normal(axes[1], ___, "log10(streams + 1)")
plt.tight_layout()
plt.show()

**❓ Pytanie:** Jak wygląda rozkład `streams`, a jak `log10(streams)`? Dlaczego w modelu 1 lepiej modelować `log10(streams)`?

**Odpowiedź:** …

### ✏️ Zadanie 6: `viral` (model 2)

Policz liczebność i procent każdej klasy.

In [ ]:
counts = df["viral"].___.sort_index()
pct = counts / ___ * 100

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(["Nie viral (0)", "Viral (1)"], counts.values)
for bar, c, p in zip(bars, counts.values, pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{c}\n({p:.1f}%)",
            ha="center", va="bottom")
ax.set_title("Rozkład zmiennej viral")
ax.set_ylabel("Liczba piosenek")
ax.margins(y=0.15)
plt.show()

**❓ Pytanie:** Czy klasy są zbalansowane? Jaką trafność (accuracy) osiągnąłby model, który zawsze przewiduje „nie viral”?

**Odpowiedź:** …

### ✏️ Zadanie 7: `genre` (model 3)

In [ ]:
genre_counts = df["genre"].___
genre_pct = genre_counts / genre_counts.sum() * 100

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(genre_counts.index, genre_counts.values)
for bar, c, p in zip(bars, genre_counts.values, genre_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{c}\n({p:.1f}%)",
            ha="center", va="bottom")
ax.set_title("Liczba piosenek według gatunku")
ax.set_ylabel("Liczba piosenek")
ax.tick_params(axis="x", rotation=45)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

**❓ Pytanie:** Ile jest gatunków i czy są równoliczne? Jaka byłaby trafność modelu, który zgaduje gatunek losowo?

**Odpowiedź:** …

## 4. Zależności między zmiennymi

### ✏️ Zadanie 8: korelacje Pearsona

Policz macierz korelacji Pearsona dla cech, `log_streams` i `viral`. Używamy `log10(streams)`, bo przy surowych `streams` współczynnik Pearsona byłby zdominowany przez kilka utworów z setkami milionów odtworzeń.

In [ ]:
cols = FEATURES + ["log_streams", "viral"]
corr_all = df[cols].___
mask = np.triu(np.ones_like(corr_all, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_all, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Korelacje cech audio z log10(streams) i viral")
plt.tight_layout()
plt.show()

### ✏️ Zadanie 9: korelacje Spearmana

Tym razem użyj surowych `streams` i korelacji Spearmana.

In [ ]:
cols = FEATURES + ["streams", "viral"]
corr_all = df[cols].corr(method=___)
mask = np.triu(np.ones_like(corr_all, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_all, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Korelacje Spearmana cech audio ze streams i viral")
plt.tight_layout()
plt.show()

**❓ Pytanie:** Czym różni się korelacja Spearmana od Pearsona i dlaczego Spearmana można liczyć na surowych `streams`? Które cechy są najsilniej związane z `streams` i `viral`? Które praktycznie wcale?

**Odpowiedź:** …

### ✏️ Zadanie 10: czy `streams` i `viral` są ze sobą powiązane?

In [ ]:
display(df[["streams", "log_streams", "viral"]].___.round(3))
display(df.___["streams"].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x=___, y=___, ax=axes[0])
axes[0].set_xticks([0, 1], ["Nie viral", "Viral"])
axes[0].set_title("log(streams) według viral")

bins = np.linspace(df["log_streams"].min(), df["log_streams"].max(), 40)
for v, label in [(0, "Nie viral"), (1, "Viral")]:
    axes[1].hist(df.loc[df["viral"] == v, "log_streams"], bins=bins,
                 density=True, alpha=0.6, label=label)
axes[1].set_xlabel("log10(streams + 1)")
axes[1].set_ylabel("Gęstość")
axes[1].set_title("Rozkład log(streams) według viral")
axes[1].legend()

plt.tight_layout()
plt.show()

### ✏️ Zadanie 11: od ilu odtworzeń piosenka jest viralem?

In [ ]:
print("Min streams wśród viral:    ", df.loc[df["viral"] == 1, "streams"].___)
print("Max streams wśród nie-viral:", df.loc[df["viral"] == ___, "streams"].___)

display(df.groupby("viral")["streams"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack())

**❓ Pytanie:** Czy istnieje jeden próg liczby odtworzeń oddzielający virale od reszty? Czy w modelu 2 (`viral`) wolno użyć `streams` jako cechy, a w modelu 1 (`streams`) — `viral`? Uzasadnij.

**Odpowiedź:** …

### ✏️ Zadanie 12: gatunek a `viral`

Zbuduj tabelę krzyżową gatunek × viral i policz odsetek viralowych utworów w każdym gatunku.

In [ ]:
ct = pd.crosstab(___, ___).reindex(columns=[0, 1], fill_value=0)
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]
total = ct.sum(axis=1)
viral_pct = ___

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(ct.index, ct[0], label="Nie viral")
ax.bar(ct.index, ct[1], bottom=ct[0], label="Viral")
for i, (t, p) in enumerate(zip(total, viral_pct)):
    ax.text(i, t, f"{p:.1f}%", ha="center", va="bottom")
ax.set_title("Liczba piosenek według gatunku z udziałem viral")
ax.set_ylabel("Liczba piosenek")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

### ✏️ Zadanie 13: odsetek viral według gatunku i roku wydania

In [ ]:
viral_rate = pd.crosstab(df["genre"], df["release_year"], values=df["viral"], aggfunc=___) * 100
viral_rate = viral_rate.loc[viral_rate.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(viral_rate, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% viral"}, ax=ax)
ax.set_title("Odsetek piosenek viralowych według gatunku i roku wydania (%)")
ax.set_xlabel("Rok wydania")
ax.set_ylabel("Gatunek")
plt.tight_layout()
plt.show()

**❓ Pytanie:** Które gatunki najczęściej, a które najrzadziej stają się viralami? Czy odsetek viral zmienia się w kolejnych latach?

**Odpowiedź:** …

## 5. Wnioski

Zapisz 3–5 najważniejszych obserwacji, które warto uwzględnić przy budowie modeli 1–3 (transformacja zmiennej celu, cechy do wykluczenia, balans klas, cechy obiecujące i bezużyteczne).

**Odpowiedź:** …